# LLaMA2-7B Layer-16 `down_proj` G16 DEW PE Trace Generator

Self-contained Colab notebook for W-BFP4, Top-2 T2A-BiE4, asymmetric DEWA (`T_SKIP=9`, `T_REPLACE=3`), FP-ACC, and RTL trace generation. It writes only `input.dat`, `trace_index.csv`, and `trace_metadata.json`; no numerical golden is generated.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm numpy

import subprocess
subprocess.run(["nvidia-smi"], check=True)

## Load the verified DEW-native model and trace encoder

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
from getpass import getpass
from pathlib import Path


"""T2A-BiE4 trace encoder for the DEW processing element."""


import csv
import json
import math
import tempfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F


@dataclass(frozen=True)
class TraceConfig:
    group_size: int = 16
    exponent_bits: int = 5
    exponent_bias: int = 15
    mantissa_bits: int = 3
    sigma_k: float = 3.0
    max_outliers_per_block: int = 2
    rounding: str = "nearest_even"
    t_skip: int = 9
    t_replace: int = 3
    dew_acc_width: int = 24

    @property
    def exponent_min(self) -> int:
        return -self.exponent_bias

    @property
    def exponent_max(self) -> int:
        return (1 << self.exponent_bits) - 1 - self.exponent_bias

    @property
    def mantissa_max(self) -> int:
        return (1 << self.mantissa_bits) - 1

    @property
    def packed_mantissa_bits(self) -> int:
        return self.group_size * self.mantissa_bits

    @property
    def packed_mantissa_hex_digits(self) -> int:
        return (self.packed_mantissa_bits + 3) // 4

    def validate(self) -> None:
        if self.group_size != 16:
            raise ValueError("The DEW PE trace contract requires G16.")
        if self.exponent_bits != 5 or self.exponent_bias != 15:
            raise ValueError("The DEW PE trace contract requires biased E5 with bias 15.")
        if self.mantissa_bits != 3:
            raise ValueError("T2A-BiE4 requires a 3-bit magnitude.")
        if self.max_outliers_per_block != 2:
            raise ValueError("T2A-BiE4 encodes at most two outliers per block.")
        if not math.isfinite(self.sigma_k) or self.sigma_k < 0:
            raise ValueError("sigma_k must be finite and non-negative.")
        if self.rounding != "nearest_even":
            raise ValueError("The trace contract requires torch.round nearest-even rounding.")
        if min(self.t_skip, self.t_replace, self.dew_acc_width) <= 0:
            raise ValueError("DEW parameters must be positive.")


@dataclass(frozen=True)
class EncodedBlocks:
    encoded_exponent: torch.Tensor
    sign: torch.Tensor
    magnitude: torch.Tensor


@dataclass(frozen=True)
class EncodedActivationBlocks:
    normal_encoded_exponent: torch.Tensor
    outlier_encoded_exponent: torch.Tensor
    sign: torch.Tensor
    magnitude: torch.Tensor
    oi1: torch.Tensor
    oi2: torch.Tensor
    candidate_count: torch.Tensor
    outlier_count: torch.Tensor


def _reshape_blocks(rows: torch.Tensor, config: TraceConfig) -> tuple[torch.Tensor, int]:
    if not rows.is_floating_point():
        raise TypeError("rows must be floating-point.")
    width = rows.shape[-1]
    padding = (-width) % config.group_size
    flat = rows.reshape(-1, width).float()
    if padding:
        flat = F.pad(flat, (0, padding))
    return flat.reshape(flat.shape[0], -1, config.group_size), padding


def signed_tensor_threshold(
    tensor: torch.Tensor,
    sigma_k: float = 3.0,
    chunk_rows: int = 2048,
) -> torch.Tensor:
    """Compute signed mean plus population-standard-deviation threshold."""
    if tensor.numel() == 0:
        raise ValueError("Cannot compute a threshold for an empty tensor.")
    if chunk_rows <= 0:
        raise ValueError("chunk_rows must be positive.")
    flat = tensor.reshape(-1, tensor.shape[-1])
    running_count = 0
    running_mean = torch.zeros((), dtype=torch.float64, device=flat.device)
    running_m2 = torch.zeros((), dtype=torch.float64, device=flat.device)

    for start in range(0, flat.shape[0], chunk_rows):
        values = flat[start : start + chunk_rows].float()
        chunk_count = values.numel()
        chunk_var, chunk_mean = torch.var_mean(values, unbiased=False)
        chunk_mean = chunk_mean.to(torch.float64)
        chunk_var = chunk_var.to(torch.float64)
        if running_count == 0:
            running_count = chunk_count
            running_mean = chunk_mean
            running_m2 = chunk_var * chunk_count
            continue
        combined_count = running_count + chunk_count
        delta = chunk_mean - running_mean
        running_mean += delta * (chunk_count / combined_count)
        running_m2 += (
            chunk_var * chunk_count
            + delta.square() * running_count * chunk_count / combined_count
        )
        running_count = combined_count

    variance = (running_m2 / running_count).clamp_min(0.0)
    return (running_mean + sigma_k * torch.sqrt(variance)).to(torch.float32)


def _scale_exponent(
    max_abs: torch.Tensor,
    present: torch.Tensor,
    config: TraceConfig,
) -> torch.Tensor:
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    exponent = torch.floor(torch.log2(safe_max)) - (config.mantissa_bits - 1)
    exponent = exponent.clamp(config.exponent_min, config.exponent_max)
    return torch.where(present, exponent, torch.zeros_like(exponent))


def encode_weight_rows(rows: torch.Tensor, config: TraceConfig) -> EncodedBlocks:
    config.validate()
    blocks, _ = _reshape_blocks(rows, config)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    scale_exp = _scale_exponent(max_abs, max_abs != 0, config)
    step = torch.pow(2.0, scale_exp)
    signed_magnitude = torch.round(blocks / step).clamp(
        -config.mantissa_max, config.mantissa_max
    ).to(torch.int16)
    return EncodedBlocks(
        encoded_exponent=(scale_exp.squeeze(-1).to(torch.int16) + config.exponent_bias).to(torch.uint8),
        sign=(signed_magnitude < 0).to(torch.uint8),
        magnitude=signed_magnitude.abs().to(torch.uint8),
    )


def _select_top2(magnitude: torch.Tensor, candidate: torch.Tensor) -> torch.Tensor:
    selected = torch.zeros_like(candidate)
    remaining = candidate.clone()
    positions = torch.arange(magnitude.shape[-1], device=magnitude.device)
    positions = positions.view(*([1] * (magnitude.ndim - 1)), magnitude.shape[-1])
    for _ in range(2):
        has_candidate = remaining.any(dim=-1, keepdim=True)
        score = magnitude.masked_fill(~remaining, float("-inf"))
        index = score.argmax(dim=-1, keepdim=True)
        picked = (positions == index) & has_candidate
        selected |= picked
        remaining &= ~picked
    return selected


def _encode_outlier_indices(outlier: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    flat = outlier.reshape(-1, outlier.shape[-1]).cpu()
    oi1 = torch.full((flat.shape[0],), 0xF, dtype=torch.uint8)
    oi2 = torch.full((flat.shape[0],), 0xF, dtype=torch.uint8)
    for row_index, row in enumerate(flat):
        indices = torch.nonzero(row, as_tuple=False).flatten().tolist()
        if len(indices) == 1:
            oi1[row_index] = indices[0]
            oi2[row_index] = 0
        elif len(indices) == 2:
            if 0 in indices:
                indices.remove(0)
                indices.insert(0, 0)
            oi1[row_index], oi2[row_index] = indices
        elif len(indices) > 2:
            raise AssertionError("Top-2 outlier occupancy exceeded.")
    shape = outlier.shape[:-1]
    return oi1.reshape(shape), oi2.reshape(shape)


def encode_activation_rows(
    rows: torch.Tensor,
    threshold: torch.Tensor | float,
    config: TraceConfig,
) -> EncodedActivationBlocks:
    config.validate()
    blocks, _ = _reshape_blocks(rows, config)
    magnitude = blocks.abs()
    candidate = magnitude > torch.as_tensor(threshold, device=blocks.device)
    outlier = _select_top2(magnitude, candidate)
    normal = ~outlier

    normal_max = torch.where(normal, magnitude, 0.0).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_present = normal.any(dim=-1, keepdim=True)
    outlier_present = outlier.any(dim=-1, keepdim=True)
    normal_scale = _scale_exponent(normal_max, normal_present, config)
    outlier_scale = _scale_exponent(outlier_max, outlier_present, config)
    outlier_scale = torch.where(outlier_present, outlier_scale, normal_scale)
    selected_scale = torch.where(outlier, outlier_scale, normal_scale)
    signed_magnitude = torch.round(blocks / torch.pow(2.0, selected_scale)).clamp(
        -config.mantissa_max, config.mantissa_max
    ).to(torch.int16)
    oi1, oi2 = _encode_outlier_indices(outlier)
    return EncodedActivationBlocks(
        normal_encoded_exponent=(
            normal_scale.squeeze(-1).to(torch.int16) + config.exponent_bias
        ).to(torch.uint8),
        outlier_encoded_exponent=(
            outlier_scale.squeeze(-1).to(torch.int16) + config.exponent_bias
        ).to(torch.uint8),
        sign=(signed_magnitude < 0).to(torch.uint8),
        magnitude=signed_magnitude.abs().to(torch.uint8),
        oi1=oi1,
        oi2=oi2,
        candidate_count=candidate.sum(dim=-1).to(torch.uint8),
        outlier_count=outlier.sum(dim=-1).to(torch.uint8),
    )


def pack_sign(sign: torch.Tensor) -> np.ndarray:
    lanes = sign.shape[-1]
    powers = torch.bitwise_left_shift(
        torch.ones(lanes, dtype=torch.int64, device=sign.device),
        torch.arange(lanes, dtype=torch.int64, device=sign.device),
    )
    return (sign.to(torch.int64) * powers).sum(dim=-1).cpu().numpy().astype(np.uint16)


def pack_magnitude(magnitude: torch.Tensor, mantissa_bits: int) -> np.ndarray:
    rows = magnitude.detach().cpu().reshape(-1, magnitude.shape[-1]).numpy()
    packed = np.empty(rows.shape[0], dtype=object)
    for row_index, row in enumerate(rows):
        value = 0
        for lane, sample in enumerate(row):
            value |= int(sample) << (lane * mantissa_bits)
        packed[row_index] = value
    return packed.reshape(magnitude.shape[:-1])


def unpack_sign(packed: np.ndarray, lanes: int) -> np.ndarray:
    shifts = np.arange(lanes, dtype=np.uint64)
    return ((packed.astype(np.uint64)[..., None] >> shifts) & 1).astype(np.uint8)


def unpack_magnitude(packed: np.ndarray, lanes: int, mantissa_bits: int) -> np.ndarray:
    flat = packed.reshape(-1)
    unpacked = np.empty((flat.shape[0], lanes), dtype=np.uint8)
    mask = (1 << mantissa_bits) - 1
    for row_index, value in enumerate(flat):
        for lane in range(lanes):
            unpacked[row_index, lane] = (int(value) >> (lane * mantissa_bits)) & mask
    return unpacked.reshape(*packed.shape, lanes)


def _hex_line(
    control: tuple[int, int, int],
    weight: tuple[int, int, int],
    activation: tuple[int, int, int, int, int, int],
    magnitude_hex_digits: int,
) -> str:
    acc_clear, weight_load, in_valid = control
    w_sign, w_exp, w_magnitude = weight
    a_sign, a_exp, oa_exp, a_magnitude, oi1, oi2 = activation
    return (
        f"{acc_clear:x} {weight_load:x} {in_valid:x} "
        f"{w_sign:04x} {w_exp:02x} {w_magnitude:0{magnitude_hex_digits}x} "
        f"{a_sign:04x} {a_exp:02x} {oa_exp:02x} "
        f"{a_magnitude:0{magnitude_hex_digits}x} {oi1:01x} {oi2:01x}\n"
    )


def _validate_roundtrip(name: str, encoded: EncodedBlocks, config: TraceConfig) -> None:
    sign_packed = pack_sign(encoded.sign)
    magnitude_packed = pack_magnitude(encoded.magnitude, config.mantissa_bits)
    np.testing.assert_array_equal(
        unpack_sign(sign_packed, config.group_size), encoded.sign.cpu().numpy()
    )
    np.testing.assert_array_equal(
        unpack_magnitude(magnitude_packed, config.group_size, config.mantissa_bits),
        encoded.magnitude.cpu().numpy(),
    )
    if int(encoded.encoded_exponent.min()) < 0 or int(encoded.encoded_exponent.max()) >= 32:
        raise AssertionError(f"{name}: exponent outside biased E5.")


def write_trace_artifacts(
    output_dir: Path,
    raw_weights: torch.Tensor,
    raw_activations: torch.Tensor,
    activation_threshold: torch.Tensor | float,
    channel_indices: list[int],
    token_positions: list[int],
    token_ids: list[int],
    metadata: dict[str, Any],
    config: TraceConfig,
) -> dict[str, Any]:
    config.validate()
    if raw_weights.ndim != 2 or raw_activations.ndim != 2:
        raise ValueError("Weight and activation selections must be rank-2 tensors.")
    if raw_weights.shape[1] != raw_activations.shape[1]:
        raise ValueError("Weight and activation K dimensions differ.")
    if raw_weights.shape[1] % config.group_size:
        raise ValueError("The K dimension must be divisible by G16.")
    if raw_weights.shape[0] != len(channel_indices):
        raise ValueError("Weight rows do not match channel indices.")
    if raw_activations.shape[0] != len(token_positions) or len(token_ids) != len(token_positions):
        raise ValueError("Activation rows do not match token metadata.")

    weights = raw_weights.detach().cpu().contiguous()
    activations = raw_activations.detach().cpu().contiguous()
    encoded_w = encode_weight_rows(weights, config)
    encoded_a = encode_activation_rows(activations, activation_threshold, config)
    _validate_roundtrip("weight", encoded_w, config)
    _validate_roundtrip(
        "activation",
        EncodedBlocks(encoded_a.normal_encoded_exponent, encoded_a.sign, encoded_a.magnitude),
        config,
    )

    w_sign = pack_sign(encoded_w.sign)
    w_magnitude = pack_magnitude(encoded_w.magnitude, config.mantissa_bits)
    a_sign = pack_sign(encoded_a.sign)
    a_magnitude = pack_magnitude(encoded_a.magnitude, config.mantissa_bits)
    w_exp = encoded_w.encoded_exponent.cpu().numpy()
    a_exp = encoded_a.normal_encoded_exponent.cpu().numpy()
    oa_exp = encoded_a.outlier_encoded_exponent.cpu().numpy()
    oi1 = encoded_a.oi1.cpu().numpy()
    oi2 = encoded_a.oi2.cpu().numpy()

    output_dir.mkdir(parents=True, exist_ok=True)
    for name in ("input.dat", "trace_index.csv", "trace_metadata.json"):
        path = output_dir / name
        if path.exists():
            path.unlink()

    token_count = activations.shape[0]
    channel_count = weights.shape[0]
    block_count = weights.shape[1] // config.group_size
    dot_product_count = token_count * channel_count
    logical_records = dot_product_count * (block_count + 1)
    man_hex = config.packed_mantissa_hex_digits
    record = 0
    with (output_dir / "input.dat").open("w", encoding="ascii", newline="\n") as input_file, (
        output_dir / "trace_index.csv"
    ).open("w", encoding="utf-8", newline="") as index_file:
        index_writer = csv.writer(index_file)
        index_writer.writerow(
            [
                "token_slot", "token_position", "token_id", "channel_slot",
                "output_channel", "setup_record", "first_valid_record", "final_record",
            ]
        )
        for token_slot in range(token_count):
            for channel_slot in range(channel_count):
                setup_record = record
                input_file.write(
                    _hex_line(
                        (1, 1, 0),
                        (
                            int(w_sign[channel_slot, 0]),
                            int(w_exp[channel_slot, 0]),
                            int(w_magnitude[channel_slot, 0]),
                        ),
                        (0, config.exponent_bias, config.exponent_bias, 0, 0xF, 0xF),
                        man_hex,
                    )
                )
                record += 1
                for block in range(block_count):
                    next_weight = min(block + 1, block_count - 1)
                    input_file.write(
                        _hex_line(
                            (0, int(block + 1 < block_count), 1),
                            (
                                int(w_sign[channel_slot, next_weight]),
                                int(w_exp[channel_slot, next_weight]),
                                int(w_magnitude[channel_slot, next_weight]),
                            ),
                            (
                                int(a_sign[token_slot, block]),
                                int(a_exp[token_slot, block]),
                                int(oa_exp[token_slot, block]),
                                int(a_magnitude[token_slot, block]),
                                int(oi1[token_slot, block]),
                                int(oi2[token_slot, block]),
                            ),
                            man_hex,
                        )
                    )
                    record += 1
                index_writer.writerow(
                    [
                        token_slot, token_positions[token_slot], token_ids[token_slot],
                        channel_slot, channel_indices[channel_slot], setup_record,
                        setup_record + 1, record - 1,
                    ]
                )

    if record != logical_records:
        raise AssertionError(f"Wrote {record} records, expected {logical_records}.")
    candidate_histogram = torch.bincount(
        encoded_a.candidate_count.flatten().to(torch.int64), minlength=config.group_size + 1
    ).tolist()
    outlier_histogram = torch.bincount(
        encoded_a.outlier_count.flatten().to(torch.int64), minlength=3
    ).tolist()
    complete_metadata = {
        **metadata,
        "trace_format": "dew_pe_t2a_bie4_cycle_input_v1",
        "trace_config": asdict(config),
        "weight_format": "W-BFP4 G16, 1 sign plus 3-bit magnitude plus biased-E5 scale",
        "activation_format": "T2A-BiE4 G16, Top-2 capped, dual biased-E5 scales",
        "threshold_contract": {
            "formula": "mean(X) + 3 * std(X)",
            "statistics_domain": "signed complete nn.Linear input tensor",
            "std_correction": 0,
            "comparison": "candidate iff abs(X) > threshold",
            "value": float(torch.as_tensor(activation_threshold).cpu()),
        },
        "outlier_index_encoding": {
            "zero": [15, 15], "one": ["oi1", 0], "two": ["oi1", "oi2"],
            "lane_zero_rule": "lane 0 is oi1 when two outliers are encoded",
        },
        "lane_packing": "lane 0 occupies the least-significant bits",
        "weight_timing": "valid block uses registered weight; weight fields preload the next block",
        "tensor_shapes": {"weight": list(weights.shape), "activation": list(activations.shape)},
        "token_positions": token_positions,
        "token_ids": token_ids,
        "output_channels": channel_indices,
        "blocks_per_dot_product": block_count,
        "records_per_dot_product": block_count + 1,
        "dot_product_count": dot_product_count,
        "valid_records": dot_product_count * block_count,
        "logical_records": logical_records,
        "candidate_outliers_per_block_histogram": candidate_histogram,
        "encoded_outliers_per_block_histogram": outlier_histogram,
        "pack_unpack_self_check": "pass",
        "numerical_golden_generated": False,
    }
    (output_dir / "trace_metadata.json").write_text(
        json.dumps(complete_metadata, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    return complete_metadata


def run_self_test() -> None:
    config = TraceConfig()
    weights = torch.tensor([[0.0, -1.0, 0.5, 1.5] * 4], dtype=torch.float32)
    activations = torch.tensor(
        [
            [0.0] * 16,
            [5.0, 0.0, 0.0, 0.0] + [0.25] * 12,
            [5.0, 4.0, 3.0, 0.0] + [0.25] * 12,
            [0.0, 5.0, 5.0, 0.0] + [0.25] * 12,
        ],
        dtype=torch.float32,
    )
    encoded = encode_activation_rows(activations, 2.0, config)
    assert encoded.outlier_count.flatten().tolist() == [0, 1, 2, 2]
    assert (encoded.oi1.flatten().tolist(), encoded.oi2.flatten().tolist()) == (
        [15, 0, 0, 1], [15, 0, 1, 2]
    )
    encoded_w = encode_weight_rows(weights, config)
    _validate_roundtrip("weight", encoded_w, config)

    with tempfile.TemporaryDirectory() as temp_dir:
        metadata = write_trace_artifacts(
            Path(temp_dir), weights, activations[:1], 2.0, [0], [128], [42],
            {"self_test": True}, config,
        )
        lines = (Path(temp_dir) / "input.dat").read_text(encoding="ascii").splitlines()
        assert len(lines) == 2
        assert all(len(line.split()) == 12 for line in lines)
        assert lines[0].split()[-2:] == ["f", "f"]
        assert metadata["logical_records"] == 2
    print("[PASS] DEW PE T2A-BiE4 packing, OI encoding, and scheduling self-test")




import gc
import json
import math
import os
import platform
import time
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import triton
import triton.language as tl
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'meta-llama/Llama-2-7b-hf'
DATASET_ID = 'Salesforce/wikitext'
DATASET_CONFIG = 'wikitext-2-raw-v1'
SPLIT = 'test'
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = 'non_overlapping_2048_drop_remainder'

FP16_BASELINE_PPL = 5.472103118896484
TOP2_BASELINE_PPL = 6.0155558586120605
TOP2_BASELINE_RESULT = (
    'results/ppl/activation-bie-top2/llama2-7b/'
    'w-bfp4-a-bie4-top2cap-g16-signed-mu3sigma-no-lm-head-s2048.json'
)
EXPECTED_LINEAR_LAYERS = 224
EXPECTED_EVALUATED_BLOCKS = 166
EXPECTED_LOSS_TOKENS = 339_802
EXPECTED_WEIGHT_VALUES = 6_476_005_376
EXPECTED_ACTIVATION_VALUES = 387_117_481_984
T_SKIP_BITS = 9
T_REPLACE_SWEEP = (8, 7, 6, 5, 4, 3, 2)
if (
    T_SKIP_BITS <= 0
    or not T_REPLACE_SWEEP
    or any(threshold <= 0 for threshold in T_REPLACE_SWEEP)
):
    raise ValueError('All asymmetric DEWA thresholds must be positive.')
T_REPLACE_TAG = '-'.join(
    str(threshold) for threshold in T_REPLACE_SWEEP
)


@dataclass(frozen=True)
class HybridConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 3
    rounding: str = 'nearest'
    activation_threshold_method: str = 'signed_mean_plus_sigma_k_std'
    sigma_k: float = 3.0
    max_outliers_per_block: int = 2
    topk_tie_break: str = 'lowest_k_index'
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError('This notebook is fixed to Group-16.')
        if self.shared_exponent_bits != 5:
            raise ValueError('This notebook is fixed to signed E5 shared exponents.')
        if self.mantissa_bits != 3:
            raise ValueError('BFP4/BiE4 require 1 sign bit + 3 magnitude bits.')
        if self.rounding != 'nearest':
            raise ValueError('This notebook is fixed to nearest rounding.')
        if self.activation_threshold_method != 'signed_mean_plus_sigma_k_std':
            raise ValueError('Unexpected threshold method.')
        if not math.isfinite(self.sigma_k) or self.sigma_k < 0:
            raise ValueError('sigma_k must be finite and non-negative.')
        if self.max_outliers_per_block != 2:
            raise ValueError('This notebook is fixed to at most two outliers per block.')
        if self.topk_tie_break != 'lowest_k_index':
            raise ValueError('Unexpected Top-2 tie-break policy.')
        if min(self.weight_chunk_rows, self.activation_chunk_rows) <= 0:
            raise ValueError('Chunk sizes must be positive.')


@dataclass(frozen=True)
class DEWAConfig:
    skip_threshold_bits: int = 9
    replace_threshold_bits: int = 8
    enabled: bool = True
    exponent_source: str = 'current_numeric_accumulator_leading_exponent'

    def validate(self):
        if self.skip_threshold_bits <= 0:
            raise ValueError('skip_threshold_bits must be positive.')
        if self.replace_threshold_bits <= 0:
            raise ValueError('replace_threshold_bits must be positive.')
        if self.exponent_source != 'current_numeric_accumulator_leading_exponent':
            raise ValueError('DEWA must use the current numeric accumulator exponent.')


@dataclass(frozen=True)
class KernelConfig:
    group_k: int = 16
    block_m: int = 32
    block_n: int = 64
    num_warps: int = 4
    num_stages: int = 2

    def validate(self):
        if self.group_k != 16:
            raise ValueError('Kernel group_k must match BiE Group-16.')
        if self.group_k % 8:
            raise ValueError('Packed tags require group_k divisible by 8.')


FORMAT = HybridConfig()
KERNEL = KernelConfig()
FORMAT.validate()
KERNEL.validate()

RESULT_DIR = Path(
    'results/ppl/top2-asymmetric-dewa-fpacc/llama2-7b'
)
ZIP_PATH = Path(
    'artifacts/colab-downloads/'
    'w-bfp4-a-bie4-top2cap-asym-dewa-fpacc-'
    f'tskip{T_SKIP_BITS}-treplace{T_REPLACE_TAG}-'
    's2048-results.zip'
)
PRIVATE_VALUE_BITS = 1 + FORMAT.mantissa_bits
WEIGHT_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS + FORMAT.shared_exponent_bits / FORMAT.block_size
)
ACTIVATION_BITS_PER_VALUE = (
    PRIVATE_VALUE_BITS
    + 1
    + 2 * FORMAT.shared_exponent_bits / FORMAT.block_size
)

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False

print(f'Hybrid config: {FORMAT}')
print(f'Kernel config: {KERNEL}')
print(
    'Asymmetric DEWA thresholds: '
    f'T_skip={T_SKIP_BITS}, '
    f'T_replace={T_REPLACE_SWEEP}'
)
print('Exception path: every nonzero partial goes directly to FP32 FP-ACC')
print(f'Weight storage: {WEIGHT_BITS_PER_VALUE:.4f} bits/value')
print(f'Activation storage: {ACTIVATION_BITS_PER_VALUE:.4f} bits/value')
if not torch.cuda.is_available():
    print('CUDA is unavailable: CPU reference tests can run, but Triton/PPL cannot.')


ACTIVATION_COUNT_NAMES = (
    "total_values",
    "candidate_values",
    "encoded_outlier_values",
    "demoted_values",
    "total_blocks",
    "candidate_affected_blocks",
    "encoded_affected_blocks",
    "cap_triggered_blocks",
    "encoded_normal_only_blocks",
    "encoded_outlier_only_blocks",
    "encoded_mixed_blocks",
)
WEIGHT_COUNT_NAMES = ("total_values", "total_blocks")


def _empty_activation_counts(device):
    return torch.zeros(
        len(ACTIVATION_COUNT_NAMES), dtype=torch.int64, device=device
    )


def _empty_weight_counts(device):
    return torch.zeros(
        len(WEIGHT_COUNT_NAMES), dtype=torch.int64, device=device
    )


def _rate(numerator, denominator):
    numerator = int(numerator)
    denominator = int(denominator)
    return {
        "numerator": numerator,
        "denominator": denominator,
        "rate": None if denominator == 0 else numerator / denominator,
    }


def _counts_to_dict(counts, names):
    values = counts.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(names, values)}


def _histogram_percentile(histogram, q, first_bin=0):
    if not 0.0 <= q <= 1.0:
        raise ValueError("q must be in [0, 1].")
    total = sum(histogram[first_bin:])
    if total == 0:
        return None
    rank = max(1, math.ceil(q * total))
    cumulative = 0
    for value in range(first_bin, len(histogram)):
        cumulative += histogram[value]
        if cumulative >= rank:
            return value
    raise RuntimeError("Histogram percentile closure failed.")


def _summarize_activation(
    counts, candidate_histogram, encoded_histogram, include_tail
):
    candidate_histogram = [int(value) for value in candidate_histogram]
    encoded_histogram = [int(value) for value in encoded_histogram]
    total_blocks = counts["total_blocks"]
    candidate_values = counts["candidate_values"]
    encoded_values = counts["encoded_outlier_values"]
    demoted_values = counts["demoted_values"]
    candidate_affected = counts["candidate_affected_blocks"]
    encoded_affected = counts["encoded_affected_blocks"]
    cap_triggered = counts["cap_triggered_blocks"]

    expected_length = FORMAT.block_size + 1
    if len(candidate_histogram) != expected_length:
        raise RuntimeError("Unexpected candidate histogram length.")
    if len(encoded_histogram) != expected_length:
        raise RuntimeError("Unexpected encoded histogram length.")
    if sum(candidate_histogram) != total_blocks:
        raise RuntimeError("Candidate histogram does not close over blocks.")
    if sum(encoded_histogram) != total_blocks:
        raise RuntimeError("Encoded histogram does not close over blocks.")
    if sum(
        index * value for index, value in enumerate(candidate_histogram)
    ) != candidate_values:
        raise RuntimeError("Candidate histogram does not close over values.")
    if sum(
        index * value for index, value in enumerate(encoded_histogram)
    ) != encoded_values:
        raise RuntimeError("Encoded histogram does not close over values.")
    if candidate_values - encoded_values != demoted_values:
        raise RuntimeError("Candidate/encoded/demoted value closure failed.")
    if candidate_histogram[0] != total_blocks - candidate_affected:
        raise RuntimeError("Candidate zero-bin closure failed.")
    if sum(candidate_histogram[1:]) != candidate_affected:
        raise RuntimeError("Candidate affected-block closure failed.")
    if encoded_histogram[0] != counts["encoded_normal_only_blocks"]:
        raise RuntimeError("Encoded zero-bin closure failed.")
    if sum(encoded_histogram[1:]) != encoded_affected:
        raise RuntimeError("Encoded affected-block closure failed.")
    if candidate_affected != encoded_affected:
        raise RuntimeError("A nonempty candidate set must encode an outlier.")
    if sum(
        candidate_histogram[FORMAT.max_outliers_per_block + 1 :]
    ) != cap_triggered:
        raise RuntimeError("Cap-triggered block closure failed.")
    if sum(encoded_histogram[FORMAT.max_outliers_per_block + 1 :]) != 0:
        raise RuntimeError("Encoded occupancy exceeds Top-2.")
    if (
        counts["encoded_outlier_only_blocks"]
        + counts["encoded_mixed_blocks"]
        != encoded_affected
    ):
        raise RuntimeError("Encoded affected-block partition failed.")

    max_candidate = max(
        (index for index, value in enumerate(candidate_histogram) if value),
        default=None,
    )
    max_encoded = max(
        (index for index, value in enumerate(encoded_histogram) if value),
        default=None,
    )
    percentiles = (
        ("50", 0.50),
        ("90", 0.90),
        ("95", 0.95),
        ("99", 0.99),
        ("99_9", 0.999),
        ("99_99", 0.9999),
    )
    summary = {
        "counts": counts,
        "candidate_histogram_semantics": (
            "index k = blocks with exactly k threshold candidates before capping"
        ),
        "candidate_outliers_per_block_histogram": candidate_histogram,
        "encoded_histogram_semantics": (
            "index k = blocks with exactly k encoded outliers after Top-2 capping"
        ),
        "encoded_outliers_per_block_histogram": encoded_histogram,
        "max_candidate_outliers_per_block": max_candidate,
        "max_encoded_outliers_per_block": max_encoded,
        "nearest_rank_percentiles": {
            "candidate_all_blocks": {
                f"p{label}": _histogram_percentile(candidate_histogram, q)
                for label, q in percentiles
            },
            "candidate_affected_blocks_only": {
                f"p{label}": _histogram_percentile(
                    candidate_histogram, q, first_bin=1
                )
                for label, q in percentiles
            },
            "encoded_all_blocks": {
                f"p{label}": _histogram_percentile(encoded_histogram, q)
                for label, q in percentiles
            },
        },
        "rates": {
            "candidate_value_rate": _rate(
                candidate_values, counts["total_values"]
            ),
            "encoded_outlier_value_rate": _rate(
                encoded_values, counts["total_values"]
            ),
            "demoted_value_rate": _rate(
                demoted_values, counts["total_values"]
            ),
            "demoted_fraction_of_candidates": _rate(
                demoted_values, candidate_values
            ),
            "candidate_affected_block_rate": _rate(
                candidate_affected, total_blocks
            ),
            "encoded_affected_block_rate": _rate(
                encoded_affected, total_blocks
            ),
            "cap_triggered_block_rate": _rate(cap_triggered, total_blocks),
        },
    }
    if include_tail:
        summary["candidate_tail_probabilities"] = [
            {
                "minimum_candidates": minimum,
                **_rate(sum(candidate_histogram[minimum:]), total_blocks),
            }
            for minimum in range(1, FORMAT.block_size + 1)
        ]
    return summary


@torch.no_grad()
def _signed_tensor_threshold(tensor, sigma_k, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.numel() == 0:
        raise ValueError("Cannot compute a threshold for an empty tensor.")
    if chunk_rows <= 0:
        raise ValueError("chunk_rows must be positive.")

    running_count = 0
    running_mean = torch.zeros((), dtype=torch.float64, device=flat.device)
    running_m2 = torch.zeros((), dtype=torch.float64, device=flat.device)

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        values = flat[start:end].float()
        chunk_count = values.numel()
        chunk_var, chunk_mean = torch.var_mean(values, unbiased=False)
        chunk_mean = chunk_mean.to(torch.float64)
        chunk_var = chunk_var.to(torch.float64)

        if running_count == 0:
            running_mean = chunk_mean
            running_m2 = chunk_var * chunk_count
            running_count = chunk_count
            continue

        combined_count = running_count + chunk_count
        delta = chunk_mean - running_mean
        running_mean = running_mean + delta * (chunk_count / combined_count)
        running_m2 = (
            running_m2
            + chunk_var * chunk_count
            + delta.square() * running_count * chunk_count / combined_count
        )
        running_count = combined_count

    variance = (running_m2 / running_count).clamp_min(0.0)
    threshold = running_mean + sigma_k * torch.sqrt(variance)
    return threshold.to(torch.float32)


def _shared_exponent(max_abs, present, config):
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    exponent = torch.floor(torch.log2(safe_max))
    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    exponent = exponent.clamp(exp_min, exp_max)
    return torch.where(present, exponent, torch.zeros_like(exponent))


def _pack_bits_last_dim(mask):
    width = mask.shape[-1]
    padding = (-width) % 8
    if padding:
        mask = F.pad(mask, (0, padding), value=False)
    grouped = mask.reshape(*mask.shape[:-1], -1, 8).to(torch.int16)
    shifts = torch.arange(8, device=mask.device, dtype=torch.int16)
    shape = (1,) * (grouped.ndim - 1) + (8,)
    return (grouped << shifts.reshape(shape)).sum(dim=-1).to(torch.uint8)


def _unpack_bits_last_dim(packed, width):
    shifts = torch.arange(8, device=packed.device, dtype=torch.int16)
    shape = (1,) * packed.ndim + (8,)
    expanded = (
        (packed.to(torch.int16).unsqueeze(-1) >> shifts.reshape(shape)) & 1
    ).to(torch.bool)
    return expanded.reshape(*packed.shape[:-1], -1)[..., :width]


@torch.no_grad()
def _quantize_weight_bfp_rows(rows, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size
    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    exponent = _shared_exponent(max_abs, max_abs != 0, config)
    step = torch.pow(2.0, exponent - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    output = (mantissa * step).reshape(flat.size(0), padded_width)
    return output[:, :width].reshape(original_shape).to(rows.dtype)


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    counts = _empty_weight_counts(weight.device)
    blocks_per_row = math.ceil(weight.size(-1) / config.block_size)

    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        rows = weight[start:end]
        rows.copy_(_quantize_weight_bfp_rows(rows, config))
        chunk_counts = torch.tensor(
            [rows.numel(), rows.size(0) * blocks_per_row],
            dtype=torch.int64,
            device=weight.device,
        )
        counts.add_(chunk_counts)
    return counts


def _select_top2_candidates(magnitude, candidate, config):
    if magnitude.shape != candidate.shape:
        raise ValueError("Magnitude and candidate masks must have the same shape.")
    if magnitude.size(-1) != config.block_size:
        raise ValueError("Top-2 selection expects G16 in the last dimension.")

    selected = torch.zeros_like(candidate)
    remaining = candidate.clone()
    positions = torch.arange(config.block_size, device=magnitude.device)
    positions = positions.view(
        *([1] * (magnitude.ndim - 1)), config.block_size
    )

    for _ in range(config.max_outliers_per_block):
        has_candidate = remaining.any(dim=-1, keepdim=True)
        score = magnitude.masked_fill(~remaining, float("-inf"))
        index = score.argmax(dim=-1, keepdim=True)
        picked = (positions == index) & has_candidate
        selected = selected | picked
        remaining = remaining & ~picked
    return selected


@torch.no_grad()
def _quantize_activation_rows_with_threshold(rows, threshold, config):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    valid = torch.ones_like(flat, dtype=torch.bool)
    if padding:
        flat = F.pad(flat, (0, padding))
        valid = F.pad(valid, (0, padding), value=False)

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    valid_blocks = valid.reshape_as(blocks)
    magnitude = blocks.abs()

    candidate = valid_blocks & (magnitude > threshold)
    outlier = _select_top2_candidates(magnitude, candidate, config)
    demoted = candidate & ~outlier
    normal = valid_blocks & ~outlier
    normal_present = normal.any(dim=-1, keepdim=True)
    outlier_present = outlier.any(dim=-1, keepdim=True)

    normal_max = torch.where(normal, magnitude, 0.0).amax(dim=-1, keepdim=True)
    outlier_max = torch.where(outlier, magnitude, 0.0).amax(dim=-1, keepdim=True)
    normal_exp = _shared_exponent(normal_max, normal_present, config)
    outlier_exp = _shared_exponent(outlier_max, outlier_present, config)
    normal_exp = torch.where(
        ~normal_present & outlier_present, outlier_exp, normal_exp
    )
    outlier_exp = torch.where(
        ~outlier_present & normal_present, normal_exp, outlier_exp
    )
    selected_exp = torch.where(outlier, outlier_exp, normal_exp)

    step = torch.pow(2.0, selected_exp - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(
        -mantissa_max, mantissa_max
    )
    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape).to(rows.dtype)
    outlier_mask = outlier.reshape(flat.size(0), padded_width)[:, :width]

    real_block = valid_blocks.any(dim=-1)
    valid_count = valid_blocks.sum(dim=-1)
    candidate_count = candidate.sum(dim=-1)
    outlier_count = outlier.sum(dim=-1)
    candidate_affected = real_block & (candidate_count > 0)
    encoded_affected = real_block & (outlier_count > 0)
    cap_triggered = real_block & (
        candidate_count > config.max_outliers_per_block
    )
    outlier_only = real_block & (outlier_count == valid_count)
    mixed = encoded_affected & ~outlier_only
    candidate_histogram = torch.bincount(
        candidate_count[real_block], minlength=config.block_size + 1
    )
    encoded_histogram = torch.bincount(
        outlier_count[real_block], minlength=config.block_size + 1
    )
    counts = torch.stack(
        (
            torch.tensor(rows.numel(), dtype=torch.int64, device=rows.device),
            candidate.sum(dtype=torch.int64),
            outlier.sum(dtype=torch.int64),
            demoted.sum(dtype=torch.int64),
            real_block.sum(dtype=torch.int64),
            candidate_affected.sum(dtype=torch.int64),
            encoded_affected.sum(dtype=torch.int64),
            cap_triggered.sum(dtype=torch.int64),
            (real_block & ~encoded_affected).sum(dtype=torch.int64),
            outlier_only.sum(dtype=torch.int64),
            mixed.sum(dtype=torch.int64),
        )
    )
    return (
        dequantized,
        outlier_mask,
        counts,
        candidate_histogram,
        encoded_histogram,
    )


@torch.no_grad()
def quantize_activation_top2(tensor, config, chunk_rows):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    threshold = _signed_tensor_threshold(flat, config.sigma_k, chunk_rows)
    output = torch.empty_like(flat)
    packed_tags = torch.empty(
        (flat.size(0), math.ceil(width / 8)),
        dtype=torch.uint8,
        device=flat.device,
    )
    counts = _empty_activation_counts(flat.device)
    candidate_histogram = torch.zeros(
        config.block_size + 1, dtype=torch.int64, device=flat.device
    )
    encoded_histogram = torch.zeros(
        config.block_size + 1, dtype=torch.int64, device=flat.device
    )
    outlier_per_k = torch.zeros(
        width, dtype=torch.int64, device=flat.device
    )
    outlier_groups = torch.zeros(
        math.ceil(width / config.block_size),
        dtype=torch.int64,
        device=flat.device,
    )

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        (
            quantized,
            outlier_mask,
            chunk_counts,
            chunk_candidate_histogram,
            chunk_encoded_histogram,
        ) = _quantize_activation_rows_with_threshold(
            flat[start:end], threshold, config
        )
        output[start:end] = quantized
        packed_tags[start:end] = _pack_bits_last_dim(outlier_mask)
        counts.add_(chunk_counts)
        candidate_histogram.add_(chunk_candidate_histogram)
        encoded_histogram.add_(chunk_encoded_histogram)
        outlier_per_k.add_(
            outlier_mask.sum(dim=0, dtype=torch.int64)
        )

        group_padding = (-width) % config.block_size
        grouped_mask = outlier_mask
        if group_padding:
            grouped_mask = F.pad(grouped_mask, (0, group_padding), value=False)
        outlier_groups.add_(
            grouped_mask.reshape(
                grouped_mask.size(0), -1, config.block_size
            ).any(dim=-1).sum(dim=0, dtype=torch.int64)
        )

    routing_meta = {
        "rows": flat.size(0),
        "width": width,
        "outlier_per_k": outlier_per_k,
        "outlier_groups": outlier_groups,
    }
    return (
        output.reshape_as(tensor),
        packed_tags,
        threshold,
        counts,
        candidate_histogram,
        encoded_histogram,
        routing_meta,
    )


KERNEL_STAT_NAMES = (
    "total_group_output_slots",
    "normal_initial_load",
    "zero_normal_partial",
    "normal_skip_new",
    "normal_replace_old",
    "normal_add",
    "normal_nonzero_decisions",
    "exception_routed_group_slots",
    "zero_exception_partial",
    "nonzero_exception_partial",
    "exception_fp_acc_adds",
    "final_dewa_nonzero_flushes",
)

PRODUCT_STAT_NAMES = (
    "total_products",
    "normal_products",
    "exception_products",
    "expected_total_group_output_slots",
    "expected_exception_routed_group_slots",
    "expected_output_slots",
)


def _empty_kernel_stats(device):
    return torch.zeros(
        len(KERNEL_STAT_NAMES), dtype=torch.int64, device=device
    )


def _empty_product_stats(device):
    return torch.zeros(
        len(PRODUCT_STAT_NAMES), dtype=torch.int64, device=device
    )


def _named_stats(tensor, names):
    values = tensor.detach().cpu().tolist()
    return {name: int(value) for name, value in zip(names, values)}


def _activation_product_partition_tensor(activation_meta, output_rows):
    activation_rows = int(activation_meta["rows"])
    width = int(activation_meta["width"])
    outlier_per_k = activation_meta["outlier_per_k"]
    exception = (outlier_per_k * output_rows).sum()
    total = torch.tensor(
        activation_rows * output_rows * width,
        dtype=torch.int64,
        device=outlier_per_k.device,
    )
    normal = total - exception
    return total, normal, exception


def _expected_slot_tensors(activation_meta, output_rows):
    activation_rows = int(activation_meta["rows"])
    outlier_groups = activation_meta["outlier_groups"]
    total_group_slots = torch.tensor(
        outlier_groups.numel() * activation_rows * output_rows,
        dtype=torch.int64,
        device=outlier_groups.device,
    )
    routed_group_slots = (outlier_groups * output_rows).sum()
    output_slots = torch.tensor(
        activation_rows * output_rows,
        dtype=torch.int64,
        device=outlier_groups.device,
    )
    return total_group_slots, routed_group_slots, output_slots


def _leading_exponent(values):
    nonzero = values != 0
    exponent = torch.floor(
        torch.log2(
            values.abs().clamp_min(torch.finfo(torch.float32).tiny)
        )
    )
    return exponent, nonzero


def _dewa_reference_update(
    accumulator,
    partial,
    skip_threshold_bits,
    replace_threshold_bits,
    enabled,
):
    old_exponent, old_nonzero = _leading_exponent(accumulator)
    new_exponent, new_nonzero = _leading_exponent(partial)
    both = old_nonzero & new_nonzero
    delta = new_exponent - old_exponent

    skip_new = (
        both & enabled & (delta <= -skip_threshold_bits)
    )
    replace_old = (
        both & enabled & (delta >= replace_threshold_bits)
    )
    normal_add = both & ~skip_new & ~replace_old
    load = ~old_nonzero & new_nonzero

    updated = torch.where(load | replace_old, partial, accumulator)
    updated = torch.where(normal_add, accumulator + partial, updated)
    counts = {
        "normal_initial_load": load.sum(dtype=torch.int64),
        "zero_normal_partial": (~new_nonzero).sum(dtype=torch.int64),
        "normal_skip_new": skip_new.sum(dtype=torch.int64),
        "normal_replace_old": replace_old.sum(dtype=torch.int64),
        "normal_add": normal_add.sum(dtype=torch.int64),
        "normal_nonzero_decisions": both.sum(dtype=torch.int64),
    }
    return updated, counts


@torch.no_grad()
def reference_hybrid_linear(
    x,
    x_outlier,
    weight,
    bias,
    dewa_config,
    group_k=16,
):
    if x.ndim != 2 or weight.ndim != 2:
        raise ValueError("Reference expects flattened 2-D operands.")
    m, k = x.shape
    n, weight_k = weight.shape
    if k != weight_k or k % group_k:
        raise ValueError("K must match and be divisible by group_k.")
    dewa_config.validate()

    normal_acc = torch.zeros((m, n), dtype=torch.float32, device=x.device)
    fp_acc = torch.zeros_like(normal_acc)
    stats = _empty_kernel_stats(x.device)

    for start in range(0, k, group_k):
        end = start + group_k
        x_group = x[:, start:end].float()
        w_group = weight[:, start:end].float()
        x_tag = x_outlier[:, start:end]

        x_normal = torch.where(x_tag, 0.0, x_group)
        x_exception = torch.where(x_tag, x_group, 0.0)
        normal_partial = x_normal @ w_group.transpose(0, 1)
        exception_partial = x_exception @ w_group.transpose(0, 1)

        normal_acc, normal_counts = _dewa_reference_update(
            normal_acc,
            normal_partial,
            dewa_config.skip_threshold_bits,
            dewa_config.replace_threshold_bits,
            dewa_config.enabled,
        )
        fp_acc = fp_acc + exception_partial

        routed = x_tag.any(dim=1, keepdim=True).expand(m, n)
        exception_nonzero = exception_partial != 0
        stats[0].add_(m * n)
        for index, name in enumerate(KERNEL_STAT_NAMES[1:7], start=1):
            stats[index].add_(normal_counts[name])
        stats[7].add_(routed.sum(dtype=torch.int64))
        stats[8].add_(
            (routed & ~exception_nonzero).sum(dtype=torch.int64)
        )
        stats[9].add_(
            (routed & exception_nonzero).sum(dtype=torch.int64)
        )
        stats[10].add_(
            (routed & exception_nonzero).sum(dtype=torch.int64)
        )

    stats[11].add_((normal_acc != 0).sum(dtype=torch.int64))
    output = normal_acc + fp_acc
    if bias is not None:
        output = output + bias.float().unsqueeze(0)
    return output.to(torch.float16), stats


@triton.jit
def _top2_dewa_fpacc_kernel(
    x_ptr,
    x_tag_ptr,
    w_ptr,
    bias_ptr,
    y_ptr,
    stats_ptr,
    M,
    N,
    K,
    stride_xm,
    stride_xk,
    stride_xtm,
    stride_xtb,
    stride_wn,
    stride_wk,
    stride_ym,
    stride_yn,
    HAS_BIAS: tl.constexpr,
    DEWA_ENABLED: tl.constexpr,
    SKIP_THRESHOLD_BITS: tl.constexpr,
    REPLACE_THRESHOLD_BITS: tl.constexpr,
    GROUP_K: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    COLLECT_STATS: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, GROUP_K)
    tag_bytes = offs_k // 8
    tag_bits = offs_k % 8

    x_ptrs = (
        x_ptr
        + offs_m[:, None] * stride_xm
        + offs_k[None, :] * stride_xk
    )
    w_ptrs = (
        w_ptr
        + offs_n[:, None] * stride_wn
        + offs_k[None, :] * stride_wk
    )
    x_tag_ptrs = (
        x_tag_ptr
        + offs_m[:, None] * stride_xtm
        + tag_bytes[None, :] * stride_xtb
    )

    valid_m = offs_m < M
    valid_n = offs_n < N
    valid_output = valid_m[:, None] & valid_n[None, :]

    normal_acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    fp_acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    stat_total_slots = tl.zeros((1,), dtype=tl.int32)
    stat_normal_load = tl.zeros((1,), dtype=tl.int32)
    stat_zero_normal = tl.zeros((1,), dtype=tl.int32)
    stat_normal_skip = tl.zeros((1,), dtype=tl.int32)
    stat_normal_replace = tl.zeros((1,), dtype=tl.int32)
    stat_normal_add = tl.zeros((1,), dtype=tl.int32)
    stat_normal_decisions = tl.zeros((1,), dtype=tl.int32)
    stat_exception_routed = tl.zeros((1,), dtype=tl.int32)
    stat_zero_exception = tl.zeros((1,), dtype=tl.int32)
    stat_nonzero_exception = tl.zeros((1,), dtype=tl.int32)
    stat_exception_fp_add = tl.zeros((1,), dtype=tl.int32)

    for _ in range(0, tl.cdiv(K, GROUP_K)):
        x_block = tl.load(
            x_ptrs, mask=valid_m[:, None], other=0.0
        )
        w_block = tl.load(
            w_ptrs, mask=valid_n[:, None], other=0.0
        )
        x_tag_word = tl.load(
            x_tag_ptrs, mask=valid_m[:, None], other=0
        ).to(tl.int32)
        x_outlier = ((x_tag_word >> tag_bits[None, :]) & 1) != 0

        x_zero = tl.zeros_like(x_block)
        x_normal = tl.where(x_outlier, x_zero, x_block)
        x_exception = tl.where(x_outlier, x_block, x_zero)
        normal_partial = tl.dot(
            x_normal, tl.trans(w_block)
        ).to(tl.float32)
        exception_partial = tl.dot(
            x_exception, tl.trans(w_block)
        ).to(tl.float32)

        # Normal DEWA decisions use the current numeric accumulator state.
        old_nonzero = normal_acc != 0.0
        old_abs = tl.abs(normal_acc)
        old_exp = tl.floor(
            tl.log2(tl.where(old_nonzero, old_abs, 1.0))
        )

        new_nonzero = normal_partial != 0.0
        new_abs = tl.abs(normal_partial)
        new_exp = tl.floor(
            tl.log2(tl.where(new_nonzero, new_abs, 1.0))
        )
        normal_both = valid_output & old_nonzero & new_nonzero
        normal_delta = new_exp - old_exp
        normal_skip = (
            normal_both
            & DEWA_ENABLED
            & (normal_delta <= -SKIP_THRESHOLD_BITS)
        )
        normal_replace = (
            normal_both
            & DEWA_ENABLED
            & (normal_delta >= REPLACE_THRESHOLD_BITS)
        )
        normal_add = normal_both & ~normal_skip & ~normal_replace
        normal_load = valid_output & ~old_nonzero & new_nonzero

        exception_nonzero = exception_partial != 0.0
        fp_acc = fp_acc + exception_partial
        normal_updated = tl.where(
            normal_load | normal_replace,
            normal_partial,
            normal_acc,
        )
        normal_acc = tl.where(
            normal_add,
            normal_acc + normal_partial,
            normal_updated,
        )

        if COLLECT_STATS:
            x_group_outlier = (
                tl.sum(x_outlier.to(tl.int32), axis=1) > 0
            )
            exception_routed = (
                valid_output & x_group_outlier[:, None]
            )
            zero_exception = (
                exception_routed & ~exception_nonzero
            )
            nonzero_exception = (
                exception_routed & exception_nonzero
            )

            stat_total_slots += tl.sum(
                tl.sum(valid_output.to(tl.int32), axis=1), axis=0
            )
            stat_normal_load += tl.sum(
                tl.sum(normal_load.to(tl.int32), axis=1), axis=0
            )
            stat_zero_normal += tl.sum(
                tl.sum(
                    (
                        valid_output & ~new_nonzero
                    ).to(tl.int32),
                    axis=1,
                ),
                axis=0,
            )
            stat_normal_skip += tl.sum(
                tl.sum(normal_skip.to(tl.int32), axis=1), axis=0
            )
            stat_normal_replace += tl.sum(
                tl.sum(normal_replace.to(tl.int32), axis=1), axis=0
            )
            stat_normal_add += tl.sum(
                tl.sum(normal_add.to(tl.int32), axis=1), axis=0
            )
            stat_normal_decisions += tl.sum(
                tl.sum(normal_both.to(tl.int32), axis=1), axis=0
            )
            stat_exception_routed += tl.sum(
                tl.sum(exception_routed.to(tl.int32), axis=1),
                axis=0,
            )
            stat_zero_exception += tl.sum(
                tl.sum(zero_exception.to(tl.int32), axis=1),
                axis=0,
            )
            stat_nonzero_exception += tl.sum(
                tl.sum(nonzero_exception.to(tl.int32), axis=1),
                axis=0,
            )
            stat_exception_fp_add += tl.sum(
                tl.sum(nonzero_exception.to(tl.int32), axis=1),
                axis=0,
            )

        x_ptrs += GROUP_K * stride_xk
        w_ptrs += GROUP_K * stride_wk
        x_tag_ptrs += (GROUP_K // 8) * stride_xtb

    output = normal_acc + fp_acc
    if HAS_BIAS:
        bias = tl.load(
            bias_ptr + offs_n, mask=valid_n, other=0.0
        )
        output += bias[None, :]

    y_ptrs = (
        y_ptr
        + offs_m[:, None] * stride_ym
        + offs_n[None, :] * stride_yn
    )
    tl.store(y_ptrs, output, mask=valid_output)

    if COLLECT_STATS:
        stat_final_flush = tl.sum(
            tl.sum(
                (
                    valid_output & (normal_acc != 0.0)
                ).to(tl.int32),
                axis=1,
            ),
            axis=0,
        )
        tl.atomic_add(
            stats_ptr + 0,
            tl.sum(stat_total_slots, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 1,
            tl.sum(stat_normal_load, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 2,
            tl.sum(stat_zero_normal, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 3,
            tl.sum(stat_normal_skip, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 4,
            tl.sum(stat_normal_replace, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 5,
            tl.sum(stat_normal_add, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 6,
            tl.sum(stat_normal_decisions, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 7,
            tl.sum(stat_exception_routed, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 8,
            tl.sum(stat_zero_exception, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 9,
            tl.sum(stat_nonzero_exception, axis=0).to(tl.int64),
        )
        tl.atomic_add(
            stats_ptr + 10,
            tl.sum(stat_exception_fp_add, axis=0).to(tl.int64),
        )
        tl.atomic_add(stats_ptr + 11, stat_final_flush.to(tl.int64))


def fused_top2_dewa_linear(
    x,
    x_packed_tags,
    weight,
    bias,
    dewa_config,
    kernel_stats,
):
    dewa_config.validate()
    original_shape = x.shape
    k = original_shape[-1]
    x_2d = x.reshape(-1, k).contiguous()
    m = x_2d.size(0)
    n = weight.size(0)

    if k != weight.size(1):
        raise ValueError("Activation and weight K dimensions differ.")
    if k % KERNEL.group_k or k % 8:
        raise ValueError("K must be divisible by Group-16 and tag width.")
    if x_packed_tags.shape != (m, k // 8):
        raise ValueError("Unexpected activation tag shape.")
    if kernel_stats.numel() != len(KERNEL_STAT_NAMES):
        raise ValueError("Unexpected kernel statistics shape.")

    output = torch.empty(
        (m, n), dtype=torch.float16, device=x.device
    )
    bias_arg = bias if bias is not None else weight
    grid = (
        triton.cdiv(m, KERNEL.block_m),
        triton.cdiv(n, KERNEL.block_n),
    )
    _top2_dewa_fpacc_kernel[grid](
        x_2d,
        x_packed_tags,
        weight,
        bias_arg,
        output,
        kernel_stats,
        M=m,
        N=n,
        K=k,
        stride_xm=x_2d.stride(0),
        stride_xk=x_2d.stride(1),
        stride_xtm=x_packed_tags.stride(0),
        stride_xtb=x_packed_tags.stride(1),
        stride_wn=weight.stride(0),
        stride_wk=weight.stride(1),
        stride_ym=output.stride(0),
        stride_yn=output.stride(1),
        HAS_BIAS=bias is not None,
        DEWA_ENABLED=dewa_config.enabled,
        SKIP_THRESHOLD_BITS=(
            dewa_config.skip_threshold_bits
        ),
        REPLACE_THRESHOLD_BITS=(
            dewa_config.replace_threshold_bits
        ),
        GROUP_K=KERNEL.group_k,
        BLOCK_M=KERNEL.block_m,
        BLOCK_N=KERNEL.block_n,
        COLLECT_STATS=True,
        num_warps=KERNEL.num_warps,
        num_stages=KERNEL.num_stages,
    )
    return output.reshape(*original_shape[:-1], n)


class Top2DEWAFPACCLinear(nn.Module):
    def __init__(
        self,
        linear,
        format_config,
        dewa_config,
        weight_counts,
    ):
        super().__init__()
        self.linear = linear
        self.format_config = format_config
        self.dewa_config = dewa_config
        self.weight_counts = _counts_to_dict(
            weight_counts, WEIGHT_COUNT_NAMES
        )

        device = linear.weight.device
        self.register_buffer(
            "_activation_counts",
            _empty_activation_counts(device),
            persistent=False,
        )
        self.register_buffer(
            "_candidate_histogram",
            torch.zeros(
                format_config.block_size + 1,
                dtype=torch.int64,
                device=device,
            ),
            persistent=False,
        )
        self.register_buffer(
            "_encoded_histogram",
            torch.zeros(
                format_config.block_size + 1,
                dtype=torch.int64,
                device=device,
            ),
            persistent=False,
        )
        self.register_buffer(
            "_activation_calls",
            torch.zeros((), dtype=torch.int64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_sum",
            torch.zeros((), dtype=torch.float64, device=device),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_min",
            torch.tensor(
                float("inf"), dtype=torch.float64, device=device
            ),
            persistent=False,
        )
        self.register_buffer(
            "_activation_threshold_max",
            torch.tensor(
                float("-inf"), dtype=torch.float64, device=device
            ),
            persistent=False,
        )
        self.register_buffer(
            "_kernel_stats",
            _empty_kernel_stats(device),
            persistent=False,
        )
        self.register_buffer(
            "_product_stats",
            _empty_product_stats(device),
            persistent=False,
        )

    def reset_runtime_stats(self):
        self._activation_counts.zero_()
        self._candidate_histogram.zero_()
        self._encoded_histogram.zero_()
        self._activation_calls.zero_()
        self._activation_threshold_sum.zero_()
        self._activation_threshold_min.fill_(float("inf"))
        self._activation_threshold_max.fill_(float("-inf"))
        self._kernel_stats.zero_()
        self._product_stats.zero_()

    def set_dewa_config(self, config):
        config.validate()
        self.dewa_config = config

    def forward(self, x):
        (
            x_quantized,
            x_packed_tags,
            threshold,
            activation_counts,
            candidate_histogram,
            encoded_histogram,
            activation_meta,
        ) = quantize_activation_top2(
            x,
            self.format_config,
            self.format_config.activation_chunk_rows,
        )

        threshold64 = threshold.to(torch.float64)
        self._activation_counts.add_(activation_counts)
        self._candidate_histogram.add_(candidate_histogram)
        self._encoded_histogram.add_(encoded_histogram)
        self._activation_calls.add_(1)
        self._activation_threshold_sum.add_(threshold64)
        self._activation_threshold_min.copy_(
            torch.minimum(self._activation_threshold_min, threshold64)
        )
        self._activation_threshold_max.copy_(
            torch.maximum(self._activation_threshold_max, threshold64)
        )

        partition = _activation_product_partition_tensor(
            activation_meta, self.linear.out_features
        )
        slots = _expected_slot_tensors(
            activation_meta, self.linear.out_features
        )
        self._product_stats.add_(torch.stack((*partition, *slots)))

        return fused_top2_dewa_linear(
            x_quantized,
            x_packed_tags,
            self.linear.weight,
            self.linear.bias,
            self.dewa_config,
            self._kernel_stats,
        )

    def export_stats(self):
        calls = int(self._activation_calls.detach().cpu().item())
        if calls:
            threshold_summary = {
                "count": calls,
                "mean": float(
                    self._activation_threshold_sum.detach().cpu().item()
                    / calls
                ),
                "min": float(
                    self._activation_threshold_min.detach().cpu().item()
                ),
                "max": float(
                    self._activation_threshold_max.detach().cpu().item()
                ),
            }
        else:
            threshold_summary = {
                "count": 0,
                "mean": None,
                "min": None,
                "max": None,
            }
        activation_counts = _counts_to_dict(
            self._activation_counts, ACTIVATION_COUNT_NAMES
        )
        activation_summary = _summarize_activation(
            activation_counts,
            self._candidate_histogram.detach().cpu().tolist(),
            self._encoded_histogram.detach().cpu().tolist(),
            include_tail=False,
        )
        return {
            "weight": {"counts": self.weight_counts},
            "activation": {
                "threshold_summary": threshold_summary,
                **activation_summary,
            },
            "kernel_counts": _named_stats(
                self._kernel_stats, KERNEL_STAT_NAMES
            ),
            "product_counts": _named_stats(
                self._product_stats, PRODUCT_STAT_NAMES
            ),
        }


def replace_linear_layers(
    module, format_config, dewa_config, prefix=""
):
    replaced = {}
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if (
                full_name == "lm_head"
                and not format_config.quantize_lm_head
            ):
                continue
            if child.in_features % format_config.block_size:
                raise ValueError(
                    f"{full_name}: K must be divisible by G16."
                )
            weight_counts = quantize_weight_in_place(
                child.weight, format_config
            )
            wrapper = Top2DEWAFPACCLinear(
                child,
                format_config,
                dewa_config,
                weight_counts,
            )
            setattr(module, name, wrapper)
            replaced[full_name] = wrapper
        else:
            replaced.update(
                replace_linear_layers(
                    child,
                    format_config,
                    dewa_config,
                    full_name,
                )
            )
    return replaced


def configure_dewa(
    replaced,
    skip_threshold_bits,
    replace_threshold_bits,
):
    config = DEWAConfig(
        skip_threshold_bits=skip_threshold_bits,
        replace_threshold_bits=replace_threshold_bits,
        enabled=True,
    )
    config.validate()
    for layer in replaced.values():
        layer.set_dewa_config(config)
        layer.reset_runtime_stats()
    return config


def _add_named_counts(total, current):
    for name in total:
        total[name] += current[name]


def export_experiment_stats(replaced):
    weight_totals = {name: 0 for name in WEIGHT_COUNT_NAMES}
    activation_totals = {
        name: 0 for name in ACTIVATION_COUNT_NAMES
    }
    kernel_totals = {name: 0 for name in KERNEL_STAT_NAMES}
    product_totals = {name: 0 for name in PRODUCT_STAT_NAMES}
    candidate_histogram = [0] * (FORMAT.block_size + 1)
    encoded_histogram = [0] * (FORMAT.block_size + 1)
    threshold_sum = 0.0
    threshold_count = 0
    threshold_min = float("inf")
    threshold_max = float("-inf")
    layers = []

    for name in sorted(replaced):
        stats = replaced[name].export_stats()
        layers.append({"layer_name": name, **stats})
        _add_named_counts(weight_totals, stats["weight"]["counts"])
        _add_named_counts(
            activation_totals, stats["activation"]["counts"]
        )
        _add_named_counts(kernel_totals, stats["kernel_counts"])
        _add_named_counts(product_totals, stats["product_counts"])

        for index, value in enumerate(
            stats["activation"][
                "candidate_outliers_per_block_histogram"
            ]
        ):
            candidate_histogram[index] += value
        for index, value in enumerate(
            stats["activation"][
                "encoded_outliers_per_block_histogram"
            ]
        ):
            encoded_histogram[index] += value

        threshold = stats["activation"]["threshold_summary"]
        if threshold["count"]:
            threshold_sum += threshold["mean"] * threshold["count"]
            threshold_count += threshold["count"]
            threshold_min = min(threshold_min, threshold["min"])
            threshold_max = max(threshold_max, threshold["max"])

    threshold_summary = {
        "count": threshold_count,
        "mean": (
            None
            if threshold_count == 0
            else threshold_sum / threshold_count
        ),
        "min": None if threshold_count == 0 else threshold_min,
        "max": None if threshold_count == 0 else threshold_max,
    }
    activation_summary = _summarize_activation(
        activation_totals,
        candidate_histogram,
        encoded_histogram,
        include_tail=True,
    )

    decisions = kernel_totals["normal_nonzero_decisions"]
    normal_oow = (
        kernel_totals["normal_skip_new"]
        + kernel_totals["normal_replace_old"]
    )
    total_slots = kernel_totals["total_group_output_slots"]
    routed_slots = kernel_totals["exception_routed_group_slots"]
    nonzero_exception = kernel_totals["nonzero_exception_partial"]
    fp_adds = kernel_totals["exception_fp_acc_adds"]
    final_flushes = kernel_totals["final_dewa_nonzero_flushes"]
    total_fp_requests = fp_adds + final_flushes

    aggregate = {
        "weight": {
            "counts": weight_totals,
            "shared_exponents_per_block": 1,
        },
        "activation": {
            "threshold_summary": threshold_summary,
            **activation_summary,
        },
        "kernel_counts": kernel_totals,
        "product_counts": product_totals,
        "rates": {
            "dewa_oow_rate": _rate(normal_oow, decisions),
            "dewa_skip_new_rate": _rate(
                kernel_totals["normal_skip_new"], decisions
            ),
            "dewa_replace_old_rate": _rate(
                kernel_totals["normal_replace_old"], decisions
            ),
            "exception_product_rate": _rate(
                product_totals["exception_products"],
                product_totals["total_products"],
            ),
            "exception_routed_group_slot_rate": _rate(
                routed_slots, total_slots
            ),
            "exception_nonzero_partial_rate": _rate(
                nonzero_exception, total_slots
            ),
            "exception_fp_acc_request_rate": _rate(
                fp_adds, total_slots
            ),
            "final_dewa_flush_rate_per_output": _rate(
                final_flushes,
                product_totals["expected_output_slots"],
            ),
            "estimated_total_fp_acc_request_rate": _rate(
                total_fp_requests, total_slots
            ),
        },
    }
    return {"aggregate": aggregate, "layers": layers}


def validate_experiment_stats(stats):
    aggregate = stats["aggregate"]
    weight = aggregate["weight"]["counts"]
    activation = aggregate["activation"]
    product = aggregate["product_counts"]
    kernel = aggregate["kernel_counts"]
    errors = []

    if weight["total_blocks"] * FORMAT.block_size != weight["total_values"]:
        errors.append("weight block/value closure failed")
    if activation["counts"]["total_blocks"] * FORMAT.block_size != (
        activation["counts"]["total_values"]
    ):
        errors.append("activation block/value closure failed")
    if activation["max_encoded_outliers_per_block"] > (
        FORMAT.max_outliers_per_block
    ):
        errors.append("encoded Top-2 occupancy exceeded")

    if product["total_products"] != (
        product["normal_products"] + product["exception_products"]
    ):
        errors.append("normal/exception product closure failed")
    if product["expected_total_group_output_slots"] != (
        kernel["total_group_output_slots"]
    ):
        errors.append("total group-slot analytical/kernel mismatch")
    if product["expected_exception_routed_group_slots"] != (
        kernel["exception_routed_group_slots"]
    ):
        errors.append("routed group-slot analytical/kernel mismatch")
    if kernel["total_group_output_slots"] != (
        kernel["zero_normal_partial"]
        + kernel["normal_initial_load"]
        + kernel["normal_nonzero_decisions"]
    ):
        errors.append("normal group-slot partition failed")
    if kernel["normal_nonzero_decisions"] != (
        kernel["normal_skip_new"]
        + kernel["normal_replace_old"]
        + kernel["normal_add"]
    ):
        errors.append("DEWA decision closure failed")
    if kernel["exception_routed_group_slots"] != (
        kernel["zero_exception_partial"]
        + kernel["nonzero_exception_partial"]
    ):
        errors.append("exception routed-slot partition failed")
    if kernel["exception_fp_acc_adds"] != (
        kernel["nonzero_exception_partial"]
    ):
        errors.append("all nonzero exception partials must reach FP-Acc")
    if kernel["final_dewa_nonzero_flushes"] > (
        product["expected_output_slots"]
    ):
        errors.append("final DEWA flushes exceed output slots")

    return {"passed": not errors, "errors": errors}


run_self_test()
print("Self-contained DEW-native definitions loaded.")


## Configure the fixed Experiment 10 workload

In [ ]:
MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
DATASET_SPLIT = "test"
TARGET_LAYER = 16
TARGET_MODULE = "model.layers.16.mlp.down_proj"
CONTEXT_LENGTH = 2048
TOKEN_POSITIONS = list(range(128, 144))
OUTPUT_CHANNELS = [
    0, 273, 546, 819, 1092, 1365, 1638, 1911,
    2184, 2457, 2730, 3003, 3276, 3549, 3822, 4095,
]
TRACE_CONFIG = TraceConfig(t_skip=9, t_replace=3, dew_acc_width=24)
OUTPUT_DIR = Path("outputs") / "llama2_layer16_down_proj_g16_wbfp4_t2abie4_dew_tskip9_treplace3"
TRACE_CONFIG.validate()
assert len(TOKEN_POSITIONS) == 16
assert len(OUTPUT_CHANNELS) == 16
print(f"Output directory: {OUTPUT_DIR}")

## Extract the DEW-native activation and write the RTL trace

In [ ]:
class CaptureComplete(RuntimeError):
    pass


def get_hf_token() -> str:
    token = os.getenv("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
    if not token:
        token = getpass("HF_TOKEN: ")
    if not token:
        raise RuntimeError("HF_TOKEN is required for the gated LLaMA2 repository.")
    return token


def tensor_sha256(tensor: torch.Tensor) -> str:
    contiguous = tensor.detach().cpu().contiguous()
    return hashlib.sha256(contiguous.numpy().tobytes()).hexdigest()


@torch.inference_mode()
def extract_dew_trace() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("An NVIDIA CUDA GPU is required for DEW-native extraction.")
    torch.manual_seed(0)
    torch.backends.cuda.matmul.allow_tf32 = False
    token = get_hf_token()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
    dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT)
    text = "\n\n".join(dataset["text"])
    all_input_ids = tokenizer(text, return_tensors="pt").input_ids
    if all_input_ids.shape[1] < CONTEXT_LENGTH:
        raise RuntimeError("WikiText-2 does not contain one complete context.")
    context_ids = all_input_ids[:, :CONTEXT_LENGTH].contiguous()
    token_ids = context_ids[0, TOKEN_POSITIONS].tolist()

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map=0,
        attn_implementation="eager",
        token=token,
    )
    model.eval()
    model.config.use_cache = False
    target_linear = model.model.layers[TARGET_LAYER].mlp.down_proj
    if not isinstance(target_linear, nn.Linear):
        raise TypeError("Expected target down_proj to be nn.Linear before wrapping.")
    if target_linear.in_features != 11008:
        raise RuntimeError(f"Unexpected target K dimension: {target_linear.in_features}")
    raw_weights = target_linear.weight[OUTPUT_CHANNELS].detach().cpu().clone()

    dewa_config = DEWAConfig(
        skip_threshold_bits=TRACE_CONFIG.t_skip,
        replace_threshold_bits=TRACE_CONFIG.t_replace,
        enabled=True,
    )
    dewa_config.validate()
    selected_layers = model.model.layers[: TARGET_LAYER + 1]
    replaced = replace_linear_layers(
        selected_layers, FORMAT, dewa_config, prefix="model.layers"
    )
    wrapped_target = model.model.layers[TARGET_LAYER].mlp.down_proj
    if not isinstance(wrapped_target, Top2DEWAFPACCLinear):
        raise TypeError("Target down_proj was not wrapped by the DEW-native model.")

    captured = {}

    def capture_target_input(_module, inputs):
        activation = inputs[0]
        if activation.shape != (1, CONTEXT_LENGTH, 11008):
            raise RuntimeError(f"Unexpected target activation shape: {tuple(activation.shape)}")
        captured["threshold"] = signed_tensor_threshold(
            activation, sigma_k=TRACE_CONFIG.sigma_k
        ).detach().cpu()
        captured["activation"] = (
            activation[0, TOKEN_POSITIONS].detach().cpu().clone()
        )
        raise CaptureComplete

    hook = wrapped_target.register_forward_pre_hook(capture_target_input)
    try:
        model(context_ids.to(next(model.parameters()).device), use_cache=False)
    except CaptureComplete:
        pass
    finally:
        hook.remove()
    if set(captured) != {"threshold", "activation"}:
        raise RuntimeError("Target activation capture did not complete.")

    metadata = {
        "model": MODEL_ID,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "dataset_split": DATASET_SPLIT,
        "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
        "context_selection": "first complete tokenized context",
        "context_length": CONTEXT_LENGTH,
        "target_module": TARGET_MODULE,
        "target_layer": TARGET_LAYER,
        "upstream_execution": (
            "layers 0 through 16 use W-BFP4, Top-2 T2A-BiE4, "
            "asymmetric DEWA Tskip9/Treplace3, and direct FP-ACC"
        ),
        "selected_weight_source": "original FP16 target weights encoded as W-BFP4",
        "selected_activation_source": (
            "input to wrapped target down_proj before its local T2A-BiE quantization"
        ),
        "linear_input_features": 11008,
        "linear_output_features": 4096,
        "quantized_linear_modules_before_capture": len(replaced),
        "selection_policy": {
            "tokens": "contiguous positions 128 through 143",
            "channels": "same evenly spaced channels as Experiment 10",
        },
        "source_hashes": {
            "context_input_ids_sha256": tensor_sha256(context_ids),
            "selected_weight_fp16_sha256": tensor_sha256(raw_weights),
            "selected_activation_fp16_sha256": tensor_sha256(captured["activation"]),
        },
        "software": {
            "python": platform.python_version(),
            "pytorch": torch.__version__,
            "transformers": transformers.__version__,
            "datasets": datasets.__version__,
            "triton": triton.__version__,
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0),
            "tokenizer_class": tokenizer.__class__.__name__,
        },
    }
    complete_metadata = write_trace_artifacts(
        OUTPUT_DIR, raw_weights, captured["activation"], captured["threshold"],
        OUTPUT_CHANNELS, TOKEN_POSITIONS, token_ids, metadata, TRACE_CONFIG,
    )
    del model, raw_weights, captured
    gc.collect()
    torch.cuda.empty_cache()
    return complete_metadata


trace_metadata = extract_dew_trace()
print(json.dumps({
    "output_dir": str(OUTPUT_DIR),
    "dot_product_count": trace_metadata["dot_product_count"],
    "valid_records": trace_metadata["valid_records"],
    "logical_records": trace_metadata["logical_records"],
    "threshold": trace_metadata["threshold_contract"]["value"],
}, indent=2))

## Inspect and download the generated artifacts

In [ ]:
files = sorted(path.name for path in OUTPUT_DIR.iterdir() if path.is_file())
assert files == ["input.dat", "trace_index.csv", "trace_metadata.json"]
assert sum(1 for _ in (OUTPUT_DIR / "input.dat").open(encoding="ascii")) == 176384
print(files)
print((OUTPUT_DIR / "input.dat").read_text(encoding="ascii").splitlines()[:3])

try:
    from google.colab import files as colab_files
except ImportError:
    print(f"Artifacts remain at: {OUTPUT_DIR.resolve()}")
else:
    import shutil
    archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
    colab_files.download(archive)